# Auto Insurance Claims: Exploratory Data Analysis & Preprocessing
This notebook loads the auto insurance claims dataset, performs data cleaning, engineers custom temporal features, prevents target leakage, and preprocesses numeric and categorical features for machine learning modeling.

## Step 1: Import Libraries

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

## Step 2: Load Raw Data

In [ ]:
data_path = 'insurance_claims.csv'
df = pd.read_csv(data_path)
print(f"Loaded raw data: {df.shape[0]} rows, {df.shape[1]} columns.")

# Drop empty helper column '_c39' if it exists
if '_c39' in df.columns:
    print("Dropping empty column '_c39'")
    df = df.drop(columns=['_c39'])

## Step 3: Handle Missing Values
We check for missing values represented by `?` and normal pandas NaNs, and replace them with a distinct `'UNKNOWN'` category.

In [ ]:
# Check missing values represented by '?'
for col in df.columns:
    num_missing = (df[col] == '?').sum()
    if num_missing > 0:
        print(f"Column '{col}' has {num_missing} missing values represented as '?'")

# Replace '?' with 'UNKNOWN'
df = df.replace('?', 'UNKNOWN')

# Fill standard pandas null values (like authorities_contacted)
df = df.fillna('UNKNOWN')

print("Missing values successfully imputed.")

## Step 4: Feature Engineering
We parse the dates, calculate policy tenure in days at the time of claim incident, calculate vehicle age at claim time, and extract temporal month/day-of-week attributes.

In [ ]:
# Parse datetime
df['policy_bind_date'] = pd.to_datetime(df['policy_bind_date'])
df['incident_date'] = pd.to_datetime(df['incident_date'])

# 1. Calculate policy tenure at the time of incident
df['policy_tenure_at_incident'] = (df['incident_date'] - df['policy_bind_date']).dt.days

# 2. Calculate vehicle age at incident time
df['vehicle_age_at_incident'] = df['incident_date'].dt.year - df['auto_year']

# 3. Extract seasonal properties
df['incident_month'] = df['incident_date'].dt.month
df['incident_day_of_week'] = df['incident_date'].dt.dayofweek

print(f"Engineered features. Policy tenure min: {df['policy_tenure_at_incident'].min()} days.")

## Step 5: Prevent Target Leakage & Define Features vs Target
To prevent target leakage, we exclude sub-claims (`injury_claim`, `property_claim`, `vehicle_claim`) and post-claim determinations (`fraud_reported`). We also drop date fields and unique IDs.

In [ ]:
drop_cols = [
    'injury_claim', 'property_claim', 'vehicle_claim', 'fraud_reported',
    'policy_number', 'insured_zip', 'incident_location',
    'policy_bind_date', 'incident_date'
]
target_col = 'total_claim_amount'

X = df.drop(columns=drop_cols + [target_col])
y = df[target_col]

numerical_features = [
    'months_as_customer', 'age', 'policy_deductable', 'policy_annual_premium',
    'umbrella_limit', 'capital-gains', 'capital-loss', 'incident_hour_of_the_day',
    'number_of_vehicles_involved', 'bodily_injuries', 'witnesses', 'auto_year',
    'policy_tenure_at_incident', 'vehicle_age_at_incident', 'incident_month', 'incident_day_of_week'
]
categorical_features = [col for col in X.columns if col not in numerical_features]

print(f"Target: {target_col}")
print(f"Numerical columns ({len(numerical_features)}): {numerical_features}")
print(f"Categorical columns ({len(categorical_features)}): {categorical_features}")

## Step 6: Train-Test Split
We split the data prior to any encoding or standardization to avoid information leakage from the test fold.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set: {X_test.shape[0]} samples")

## Step 7: Preprocessing Pipeline (StandardScaler & OneHotEncoder)
We standardize the continuous features and one-hot encode categorical features. The encoders are fit only on `X_train`.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ])

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

cat_encoder = preprocessor.named_transformers_['cat']
encoded_cat_features = list(cat_encoder.get_feature_names_out(categorical_features))
feature_names = numerical_features + encoded_cat_features

print(f"Processed training size: {X_train_proc.shape}")
print(f"Total features after One-Hot Encoding: {len(feature_names)}")

## Step 8: Save Preprocessed Data to Disk

In [ ]:
out_dir = r'src/processed'
os.makedirs(out_dir, exist_ok=True)

np.save(os.path.join(out_dir, "X_train.npy"), X_train_proc)
np.save(os.path.join(out_dir, "X_test.npy"), X_test_proc)
np.save(os.path.join(out_dir, "y_train.npy"), y_train.values)
np.save(os.path.join(out_dir, "y_test.npy"), y_test.values)

with open(os.path.join(out_dir, "feature_names.txt"), "w") as f:
    f.write("\n".join(feature_names))
    
print("Numpy preprocessed arrays successfully saved.")